In [19]:
# 导入所有需要的库
import pandas as pd
import jieba
from gensim.models import Word2Vec

# 加载餐厅评价数据（注意列名要和你的train.csv一致！）
# 如果你的csv列名是comment，就用data['comment']；如果是review，就用data['review']
df = pd.read_csv("train.csv")
print("数据加载完成，前5条数据：")
print(df.head())

数据加载完成，前5条数据：
   label                                            comment
0      0                                 一如既往地好吃，希望可以开到其他城市
1      0                                  味道很不错，分量足，客人很多，满意
2      0  下雨天来的，没有想象中那么火爆。环境非常干净，古色古香的，我自己也是个做服务行业的，我都觉得...
3      0                                    真心不好吃 基本上没得好多味道
4      0              少送一个牛肉汉堡 而且也不好吃 特别是鸡肉卷 **都不想评论了 谁买谁知道


In [20]:
# 定义分词函数，同时做基础清洗
def clean_and_cut(sentence):
    # 基础文本清洗：去掉标点、特殊符号
    sentence = sentence.replace("，", "").replace("。", "").replace("！", "").replace("？", "")
    sentence = sentence.replace("!", "").replace("?", "").replace("~", "").replace("#", "")
    # jieba分词，过滤空字符
    return [word for word in jieba.lcut(sentence) if word.strip()]

# 对所有评价进行分词，生成训练语料sentences
# 注意：这里的列名要和单元格1一致！
sentences = df['comment'].apply(clean_and_cut).tolist()  # 如果列名是review，改成df['review']
print("分词完成，前2条分词结果示例：")
print(sentences[:2])
print(f"总语料条数：{len(sentences)}")

分词完成，前2条分词结果示例：
[['一如既往', '地', '好吃', '希望', '可以', '开', '到', '其他', '城市'], ['味道', '很', '不错', '分量', '足', '客人', '很多', '满意']]
总语料条数：10000


In [21]:
# ====================== Skip-Gram模型训练 ======================
# 核心参数说明：
# sg=1 → 启用Skip-gram模式（sg=0为CBOW）
# vector_size=100 → 词向量维度（50-300之间，常用100）
# window=5 → 上下文窗口大小
# min_count=1 → 过滤出现次数<1的词（保留所有词）
# workers=4 → 并行训练线程数
model = Word2Vec(
    sentences=sentences,
    sg=1,          # 关键：切换为Skip-Gram模式
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

# 保存模型（可选，方便后续复用）
model.save("word2vec_skipgram.model")
print("Skip-Gram模型训练完成！")
print(f"词汇表大小：{len(model.wv)}")

Skip-Gram模型训练完成！
词汇表大小：12096


In [22]:
# 获取"环境"的词向量
env_vector = model.wv['环境']
print("'环境'的词向量：")
print(env_vector)
print(f"\n词向量形状：{env_vector.shape}")

'环境'的词向量：
[-0.03443179 -0.06453842 -0.4695432  -0.30517596 -0.263696   -0.32462487
  0.5318006   0.63679975 -0.0873865  -0.5163801  -0.12007891  0.02612019
  0.31354958  0.53571725  0.37003043 -0.5673771   0.03681375 -0.0339193
 -0.51975405 -0.46033353 -0.06635685  0.24888198 -0.09555063 -0.09248918
 -0.13379495 -0.19207875 -0.27881876 -0.26178306 -0.4802014   0.5817649
  0.45424125 -0.16458264  0.20356813 -0.14958087 -0.32116422  0.810516
 -0.3061473  -0.03186468 -0.1258175  -0.4613421   0.20219144 -0.3801424
  0.1545133   0.32114276  0.6216674   0.10424632 -0.6803264  -0.45952424
  0.28775203  0.36368304  0.274681    0.1630777   0.04806415 -0.21881741
 -0.29417235  0.19703299  0.53747     0.31379417 -0.3669772   0.05161393
  0.26315492 -0.22988352  0.47470534 -0.2552353  -0.28331342  0.8751155
  0.17284833  0.39339676 -0.37647066 -0.03700911 -0.25188687  0.2724439
  0.34844762  0.4139977   0.46095896  0.14635183  0.3397259   0.13393517
 -0.37985823  0.25519326 -0.23559745 -0.09659501

In [23]:
similar_words = model.wv.most_similar('好吃', topn=3)
print("与'好吃'语义最接近的3个词：")
for word, sim in similar_words:
    print(f"{word}: {sim:.4f}")

与'好吃'语义最接近的3个词：
入味: 0.8978
好看: 0.8888
棒: 0.8850


In [24]:
sim_tasty = model.wv.similarity('好吃', '美味')
sim_cockroach = model.wv.similarity('好吃', '蟑螂')
print(f"'好吃' 和 '美味' 的相似度：{sim_tasty:.4f}")
print(f"'好吃' 和 '蟑螂' 的相似度：{sim_cockroach:.4f}")

'好吃' 和 '美味' 的相似度：0.8757
'好吃' 和 '蟑螂' 的相似度：0.4556


In [25]:
result = model.wv.most_similar(positive=['餐厅', '聚会'], negative=['安静'], topn=1)
print("向量运算 '餐厅+聚会-安静=' 的最相关结果：")
print(f"{result[0][0]}，相似度：{result[0][1]:.4f}")

向量运算 '餐厅+聚会-安静=' 的最相关结果：
外地，相似度：0.9595
